# Europe PMC Tools

![Europe PMC Tools](https://proto-bio.github.io/proto-assets/images/tool/europepmc/hero.png)

This notebook covers the two Europe PMC tools. `run_europepmc_search` finds papers and preprints in Europe PMC by free text or field query, and `run_europepmc_fulltext` reads an open-access article as plain-text sections. Together they take you from a design question to the exact passage, table row, or figure caption that answers it.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("europepmc")
display_overview("europepmc")
display_docs_section("europepmc", "Background")

# Europe PMC

[Europe PMC](https://europepmc.org/) is EMBL-EBI's open literature database, covering PubMed abstracts, PubMed Central full-text articles, and preprints from bioRxiv, medRxiv, and other servers. This toolkit exposes two tools over its REST API: `europepmc-search` finds papers by free text or field query and reports identifiers, citation details, and open-access status, and `europepmc-fulltext` returns an open-access article as plain-text sections that can be searched for an exact passage. Both run on CPU in process and need only network access.

[Europe PMC](https://europepmc.org/) is run by [EMBL-EBI](https://www.ebi.ac.uk/) as a partner of [PubMed Central](https://www.ncbi.nlm.nih.gov/pmc/). It indexes tens of millions of records, mostly PubMed abstracts, along with millions of full-text articles and preprints. Each record has a source code (`MED` for PubMed, `PMC` for PubMed Central, `PPR` for preprints) and an identifier within that source, and carries a PMID, PMCID, or DOI where one exists.

Full text is available for the open-access subset of PubMed Central and Europe PMC, published under licenses such as CC BY that permit text mining. Articles arrive as [JATS](https://jats.nlm.nih.gov/) XML, the standard publishing markup for journal articles, which marks the title, abstracts, nested sections, figures, and tables. Reading an article as text rather than as a PDF makes it possible to find and quote a reported value, such as a binding constant or a list of hotspot residues, exactly as the authors wrote it.

## Available tools

In [2]:
display_available_tools("europepmc")

- **`run_europepmc_fulltext()`** — Fetch open-access article full text from Europe PMC by PMCID as plain-text sections, figure and table captions, and tables, to find and quote supporting passages
- **`run_europepmc_search()`** — Search Europe PMC literature (PubMed, PMC full text, bioRxiv/medRxiv preprints) for papers with PMIDs, PMCIDs, DOIs, abstracts and open-access status

### `run_europepmc_search`

`run_europepmc_search` ranks Europe PMC records for a query written in free text or Europe PMC syntax. Each hit carries its PMID, PMCID, and DOI, citation details, whether it is open access, and its abstract. Here we look for open-access papers on de novo designed binders that report binding affinities.

In [3]:
from proto_tools import (
    EuropePMCSearchConfig,
    EuropePMCSearchInput,
    run_europepmc_search,
)

In [4]:
# Display docs
display_api_reference("europepmc", "input", "run_europepmc_search")

# Search open-access papers on de novo binder design that mention affinity
inputs = EuropePMCSearchInput(queries='"de novo" "binder design" affinity OPEN_ACCESS:y')

**Input** — `EuropePMCSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>queries</code> | <code>list[str]</code> | required | Free-text or Europe PMC query-syntax searches; a bare string is accepted |

In [5]:
# Display config docs
display_api_reference("europepmc", "config", "run_europepmc_search")

# Return the top 5 hits with abstracts | see docs above for all fields
config = EuropePMCSearchConfig(max_results=5)

**Config** — `EuropePMCSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>max_results</code> | <code>int</code> | <code>10</code> | Maximum hits returned per query |
| <code>include_abstracts</code> | <code>bool</code> | <code>True</code> | Return abstracts (core results); False for lite results |

In [6]:
# Run the search
search_output = run_europepmc_search(inputs, config)

In [7]:
# Display output docs
display_api_reference("europepmc", "output", "run_europepmc_search")

# Inspect the ranked hits
result = search_output.results[0]
print(f"{result.total_count} matching records")
for hit in result.hits:
    print(f"{hit.pmcid or hit.id}  ({hit.year})  OA={hit.is_open_access}  {hit.title[:80]}")

**Output** — `EuropePMCSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[EuropePMCSearchResult]</code> | <code>[]</code> | Results in input order |
| <code>source_url</code> | <code>str</code> | required | Europe PMC search endpoint used |

**`EuropePMCSearchResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query</code> | <code>str</code> | required | Query as sent |
| <code>total_count</code> | <code>int &#124; None</code> | <code>None</code> | Total matching records |
| <code>hits</code> | <code>list[EuropePMCSearchHit]</code> | <code>[]</code> | Hits in relevance order |
| <code>error</code> | <code>str &#124; None</code> | <code>None</code> | Why this item has no data; None when it was retrieved |

**`EuropePMCSearchHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>str</code> | required | Europe PMC record ID |
| <code>source</code> | <code>str</code> | required | Europe PMC source code (MED, PMC, PPR, ...) |
| <code>pmid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed identifier |
| <code>pmcid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed Central identifier |
| <code>doi</code> | <code>str &#124; None</code> | <code>None</code> | Digital object identifier |
| <code>title</code> | <code>str</code> | required | Publication title |
| <code>authors</code> | <code>list[str]</code> | <code>[]</code> | Author names as Europe PMC gives them |
| <code>journal</code> | <code>str &#124; None</code> | <code>None</code> | Journal or preprint server |
| <code>year</code> | <code>int &#124; None</code> | <code>None</code> | Publication year |
| <code>publication_date</code> | <code>str &#124; None</code> | <code>None</code> | First publication date (core results) |
| <code>is_open_access</code> | <code>bool</code> | <code>False</code> | Whether the article is OA |
| <code>has_full_text</code> | <code>bool</code> | <code>False</code> | Whether Europe PMC holds the full text |
| <code>cited_by_count</code> | <code>int &#124; None</code> | <code>None</code> | Citing records |
| <code>abstract</code> | <code>str &#124; None</code> | <code>None</code> | Abstract (core results) |
| <code>published_pmid</code> | <code>str &#124; None</code> | <code>None</code> | For a preprint, the PMID of its published article |
| <code>source_url</code> | <code>str</code> | required | Europe PMC page for the record |

233 matching records
PMC13654480  (2026)  OA=True  Sequence and structural determinants of efficacious de novo chimaeric antigen re
PMC13113996  (2026)  OA=True  AI-Driven De Novo Binder Design: From Structure Prediction to Closed-Loop Optimi
PMC13658209  (2026)  OA=True  Ovo, an open-source ecosystem for de novo protein design.
PMC13339307  (2026)  OA=True  De Novo Designed Minibinders Targeting the GDF15-GFRAL Axis Reverse Cancer Cache
PMC13029676  (2026)  OA=True  AI-Driven Design of Miniproteins as Potential Allosteric Modulators.


#### Preprints and Field Queries

Field codes narrow a search more reliably than extra words. `SRC:PPR` keeps only preprints, including bioRxiv and medRxiv, which makes this the keyword search for preprints. A `DOI:` query resolves a paper you already know to its PMID and PMCID.

In [8]:
preprint_output = run_europepmc_search(
    EuropePMCSearchInput(queries=['TITLE:"protein binder" AND SRC:PPR', 'DOI:"10.1038/s41586-023-06415-8"']),
    EuropePMCSearchConfig(max_results=3, include_abstracts=False),
)

for result in preprint_output.results:
    print(f"\n{result.query}  ({result.total_count} records)")
    for hit in result.hits:
        print(f"  [{hit.source}] {hit.journal}  doi={hit.doi}  pmcid={hit.pmcid}  {hit.title[:60]}")


TITLE:"protein binder" AND SRC:PPR  (16 records)
  [PPR] bioRxiv  doi=10.64898/2026.10.22.760942  pmcid=None  Projected Gradient Descent for Hallucination-Based Protein B
  [PPR] bioRxiv  doi=10.64898/2026.10.29.762730  pmcid=None  LPT: Orchestrating bounded agentic loops for automated prote
  [PPR] bioRxiv  doi=10.64898/2026.09.08.749745  pmcid=None  Specificity-driven protein binder design with Odin-Multi

DOI:"10.1038/s41586-023-06415-8"  (1 records)
  [MED] Nature  doi=10.1038/s41586-023-06415-8  pmcid=PMC10468394  De novo design of protein structure and function with RFdiff


### `run_europepmc_fulltext`

`run_europepmc_fulltext` fetches an open-access article's JATS XML and flattens it into sections: the title, abstracts, body and back-matter sections under their heading path, figure captions, and tables with cells separated by `|`. Here we read the RFdiffusion paper resolved by DOI above.

In [9]:
from proto_tools import (
    EuropePMCFullTextConfig,
    EuropePMCFullTextInput,
    run_europepmc_fulltext,
)

In [10]:
# Display docs
display_api_reference("europepmc", "input", "run_europepmc_fulltext")

# Read the RFdiffusion paper by its PMCID
fulltext_inputs = EuropePMCFullTextInput(pmcids=["PMC10468394"])

**Input** — `EuropePMCFullTextInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pmcids</code> | <code>list[str]</code> | required | PubMed Central IDs of open-access articles, e.g. 'PMC6034017' |

In [11]:
# Display config docs
display_api_reference("europepmc", "config", "run_europepmc_fulltext")

# Use default config | see docs above for all fields
fulltext_config = EuropePMCFullTextConfig()

*No config fields.*

In [12]:
# Fetch the full text
fulltext_output = run_europepmc_fulltext(fulltext_inputs, fulltext_config)

In [13]:
# Display output docs
display_api_reference("europepmc", "output", "run_europepmc_fulltext")

# Inspect the document's sections
document = fulltext_output.documents[0]
print(f"{document.title}\nPMID {document.pmid}  DOI {document.doi}  license {document.license}\n")
for section in document.sections[:12]:
    print(f"[{section.kind}] {section.heading}: {len(section.text)} chars")

**Output** — `EuropePMCFullTextOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>documents</code> | <code>list[EuropePMCFullTextDocument]</code> | <code>[]</code> | Documents in input order |
| <code>source_url</code> | <code>str</code> | required | Europe PMC REST endpoint used |

**`EuropePMCFullTextDocument`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pmcid</code> | <code>str</code> | required | PubMed Central ID |
| <code>pmid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed ID |
| <code>doi</code> | <code>str &#124; None</code> | <code>None</code> | Digital object identifier |
| <code>title</code> | <code>str &#124; None</code> | <code>None</code> | Article title |
| <code>license</code> | <code>str &#124; None</code> | <code>None</code> | License URL or statement |
| <code>sections</code> | <code>list[EuropePMCSection]</code> | <code>[]</code> | Text blocks in document order |
| <code>source_url</code> | <code>str &#124; None</code> | <code>None</code> | Full-text XML URL the text was read from |
| <code>error</code> | <code>str &#124; None</code> | <code>None</code> | Why this item has no data; None when it was retrieved |

**`EuropePMCSection`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>kind</code> | <code>Literal['title', 'abstract', 'section', 'figure', 'table']</code> | required | title, abstract, section, figure or table |
| <code>heading</code> | <code>str &#124; None</code> | <code>None</code> | Heading path, or figure/table label |
| <code>text</code> | <code>str</code> | required | Plain text; paragraphs separated by newlines |

De novo design of protein structure and function with RFdiffusion
PMID 37433327  DOI 10.1038/s41586-023-06415-8  license https://creativecommons.org/licenses/by/4.0/

[title] None: 65 chars
[abstract] Abstract: 1656 chars
[abstract] Abstract (web-summary): 245 chars
[section] Main: 8507 chars
[figure] Fig. 1: 1885 chars
[section] Unconditional protein monomer generation: 3204 chars
[figure] Fig. 2: 2133 chars
[section] Design of higher-order oligomers: 6788 chars
[figure] Fig. 3: 1707 chars
[section] Functional-motif scaffolding: 2997 chars
[figure] Fig. 4: 2252 chars
[section] Scaffolding enzyme active sites: 1037 chars


#### Finding a Supporting Passage

Because every section is plain text, a regular expression finds the sentence that reports a value. Here we pull each sentence that states a binding affinity in nanomolar, with the heading it sits under, so it can be quoted and cited verbatim.

In [14]:
import re

# Split on sentence ends (a period followed by a capital letter), then keep affinity statements in nM
sentence_end = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")
affinity = re.compile(r"\b(?:K_?[dD]|affinit\w*)\b.*\bnM\b")
for section in document.sections:
    for sentence in sentence_end.split(section.text.replace("\n", " ")):
        if affinity.search(sentence):
            print(f"[{section.heading}] {sentence[:240]}\n")

[Functional-motif scaffolding] We scaffolded the p53 helix in the presence of MDM2, so extra interactions could be designed by RFdiffusion and experimentally identified 0.5 and 0.7 nM binders (Fig. 4c,d), three orders of magnitude higher affinity than the reported 600 nM

[Fig. 4] Designs were generated with an RFdiffusion model fine-tuned on complexes. d, BLI measurements indicate high-affinity binding to MDM2 (p53_design_89, 0.7 nM; p53_design_53, 0.5 nM); the native affinity is 600 nM (ref. 42). e, Out of 95 desig

[Design of protein-binding proteins] Full BLI titrations for a subset of the designs showed nanomolar affinities with no further experimental optimization, including HA and IL-7Rɑ binders with affinities of roughly 30 nM (Fig. 6c).

[Fig. 6] Reported KD values are based on global kinetic fitting with fixed global Rmax. d, The highest affinity HA binder, HA_20, binds with a KD of 28 nM. c,d, Yellow or orange, target or hotspot residues; grey, design model; purple, AF2 pred

## Export Results

Search results and documents can be saved to JSON, and documents also to plain text for searching with other tools. Each article keeps its own copyright and license, so this example writes to a temporary directory rather than committing retrieved text.

In [15]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    search_output.export("europepmc_search", export_path=output_dir, file_format="csv")
    fulltext_output.export("europepmc_fulltext", export_path=output_dir, file_format="txt")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")

Exported ['europepmc_fulltext.txt', 'europepmc_search.csv']
